### Load librarys and heathiar

In [1]:
import os

# Point Python/rpy2 to the Folder wher your R is located  --> healthiar has to be already installed there
os.environ['R_HOME'] = r"C:\Program Files\R\R-4.4.1"

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import FloatVector, IntVector,globalenv,ListVector, DataFrame, StrVector, BoolVector
from rpy2.robjects import pandas2ri
import pandas as pd
from rpy2.robjects.conversion import localconverter
from types import SimpleNamespace




healhipy = importr("healthiar")

In [9]:
#function which converts rpy2 lists and dataframes to python lists and pandas dataframes
def rpy2_to_py(obj):
    """
    Recursive conversion
    """
    if isinstance(obj, DataFrame):
        with localconverter(ro.default_converter + pandas2ri.converter):
            return ro.conversion.rpy2py(obj)
    
    elif isinstance(obj, ListVector):
        return {name: rpy2_to_py(obj.rx2(name)) for name in obj.names}
    
    elif isinstance(obj, (IntVector, FloatVector, StrVector, BoolVector)):
        if len(obj) == 1:
            return obj[0]  # single elements
        else:
            return list(obj)  # multiple elements
    else:
        return obj  # fallback for other types


### Example 1
Use attribute_health

In [3]:

# Python-style inputs
exp_central = [20, 20]
prop_pop_exp = [0.5, 0.5]
bhd_central = [10]

# Convert to R vectors
r_exp_central = FloatVector(exp_central)
r_prop_pop_exp = FloatVector(prop_pop_exp)
r_bhd_central = FloatVector(bhd_central)

# Call healthiar function
result = healhipy.attribute_health(
    exp_central=r_exp_central,
    prop_pop_exp=r_prop_pop_exp,
    cutoff_central=5,
    rr_central=1.08,
    rr_increment=10,
    erf_shape="linear_log",
    bhd_central=r_bhd_central
)
print(result.rx2('health_main').rx2('impact'))





result.rx2('health_main')


[1] 0.927071



geo_id_micro,erf_ci,exp_ci,...,pop_fraction_type,rr_at_exp,pop_fraction
'a','central','central',...,'paf',1.102180,0.092707


In [12]:
py_result = rpy2_to_py(result)
py_result["health_detailed"]["results_raw"].T

,1
approach_risk,relative_risk
rr_increment,10
erf_shape,linear_log
prop_pop_exp,0.5
geo_id_micro,a
age_group,all
sex,all
exp_length,2
exp_category,"1, 2"
exp_type,exposure_distribution


### Example 2:
Use compare with atribute_health inputs

In [13]:
#check if compare works

# Call healthiar function
result1 = healhipy.attribute_health(
    exp_central=8.85,
    cutoff_central=5,
    rr_central=1.118,
    rr_lower = 1.060,
    rr_upper = 1.179,
    rr_increment=10,
    erf_shape="log_linear",
    bhd_central=25000,
    approach_risk = "relative_risk"
)


# Call healthiar function
result2 = healhipy.attribute_health(
    exp_central=6,
    cutoff_central=5,
    rr_central=1.118,
    rr_lower = 1.060,
    rr_upper = 1.179,
    rr_increment=10,
    erf_shape="log_linear",
    bhd_central=25000,
    approach_risk = "relative_risk"
)




As long as result1 and result2 stay in rpy2 format the can be used in healthypy
Afterwards we need to transfom it back to python

In [15]:
compared_r = rpy2_to_py(healhipy.compare(result1,result2,approach_comparison = "delta"))
compared_r["health_main"]

,geo_id_micro,erf_ci,exp_ci,bhd_ci,cutoff_ci,exp_category,sex,age_group,impact_scen_1,impact_scen_2,...,is_lifetable,pop_fraction_type,rr_at_exp_scen_1,pop_fraction_scen_1,prop_pop_exp_scen_2,exp_scen_2,rr_at_exp_scen_2,pop_fraction_scen_2,impact_scen_1_rounded,impact_scen_2_rounded
1,a,central,central,central,central,1,all,all,1050.860466,277.304018,...,False,paf,1.043879,0.042034,1.0,6,1.011217,0.011092,1051.0,277.0
2,a,lower,central,central,central,1,all,all,554.594229,145.248685,...,False,paf,1.022687,0.022184,1.0,6,1.005844,0.005810,555.0,145.0
3,a,upper,central,central,central,1,all,all,1535.722093,408.295694,...,False,paf,1.065449,0.061429,1.0,6,1.016603,0.016332,1536.0,408.0


### Access healthiar documentation

In [23]:
?healhipy.attribute_health

Signature:       healhipy.attribute_health(*args, **kwargs)
Type:            DocumentedSTFunction
String form:    
function (approach_risk = "relative_risk", exp_central, exp_lower = NULL,
           exp_upper = NULL,  <...> put_args)
           return(output)
           }
           <bytecode: 0x000001a0765f3370>
           <environment: namespace:healthiar>
           
File:            c:\users\ardigi\.conda\envs\myenv\lib\site-packages\rpy2\robjects\functions.py
Docstring:      
Wrapper around an R function.

The docstring below is built from the R documentation.

description
-----------


 This function calculates the attributable health impacts (mortality or morbidity) due to
 exposure to an environmental stressor (air pollution or noise), using either relative risk ( RR ) or absolute risk ( AR ).
 
 Arguments for both  RR & AR  pathways
 
     approach_risk 
     exp_central ,  exp_lower ,  exp_upper 
     cutoff_central ,  cutoff_lower ,  cutoff_upper 
     erf_eq_central ,  erf